# Wersja 1: klasyczna sieć sekwencyjna w TensorFlow/Keras

To podejście pokazuje klasyczny pipeline NLP oparty na sieci neuronowej:

**TextVectorization → Embedding → BiLSTM → Dense(softmax)**

Ta wersja jest świetna dydaktycznie, bo studenci widzą cały przepływ:
1. surowy tekst,
2. zamiana na indeksy tokenów,
3. embeddingi,
4. analiza sekwencji przez BiLSTM,
5. klasyfikacja do 6 emocji.

# Klasyfikacja 6 emocji w języku polskim

Ten notebook pracuje na pliku `emocje_20000.csv`, który zawiera **20 000 syntetycznych tekstów po polsku** oznaczonych jedną z 6 emocji:

- `anger`
- `disgust`
- `fear`
- `joy`
- `sadness`
- `surprise`

Kolumny w pliku:
- `text`
- `label`
- `label_id`
- `split`

Notebook można uruchomić lokalnie w Jupyter Notebook / JupyterLab.

In [ ]:
# Jeśli potrzeba, odkomentuj:
# %pip install pandas numpy scikit-learn tensorflow matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import classification_report, confusion_matrix

## 1. Konfiguracja

In [ ]:
CSV_PATH = "emocje_20000.csv"
BATCH_SIZE = 32
MAX_TOKENS = 20000
SEQUENCE_LENGTH = 40
EMBEDDING_DIM = 128
LSTM_UNITS = 64
EPOCHS = 8
RANDOM_SEED = 42

tf.keras.utils.set_random_seed(RANDOM_SEED)

## 2. Wczytanie danych

In [ ]:
df = pd.read_csv(CSV_PATH)

required_cols = {"text", "label", "label_id", "split"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Brakuje kolumn w pliku CSV: {missing}")

train_df = df[df["split"] == "train"].copy()
val_df   = df[df["split"] == "val"].copy()
test_df  = df[df["split"] == "test"].copy()

print("Rozmiary zbiorów:")
print("train:", len(train_df))
print("val:  ", len(val_df))
print("test: ", len(test_df))

display(df.head())

In [ ]:
print("Rozkład klas:")
display(df["label"].value_counts())

id_to_label = (
    df[["label_id", "label"]]
    .drop_duplicates()
    .sort_values("label_id")
    .set_index("label_id")["label"]
    .to_dict()
)

label_to_id = {v: k for k, v in id_to_label.items()}
print("Mapowanie label_to_id:", label_to_id)

## 3. Datasety TensorFlow

In [ ]:
train_ds = tf.data.Dataset.from_tensor_slices(
    (train_df["text"].astype(str).values, train_df["label_id"].values)
).shuffle(len(train_df)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices(
    (val_df["text"].astype(str).values, val_df["label_id"].values)
).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices(
    (test_df["text"].astype(str).values, test_df["label_id"].values)
).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

## 4. Wektoryzacja tekstu

In [ ]:
vectorize_layer = tf.keras.layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode="int",
    output_sequence_length=SEQUENCE_LENGTH,
    standardize="lower_and_strip_punctuation"
)

text_only_train_ds = train_ds.map(lambda x, y: x)
vectorize_layer.adapt(text_only_train_ds)

In [ ]:
sample_texts = train_df["text"].head(3).tolist()
vectorized = vectorize_layer(tf.constant(sample_texts))

for text, vec in zip(sample_texts, vectorized.numpy()):
    print("TEKST:", text)
    print("WEKTOR:", vec[:20], "...")
    print("-" * 80)

## 5. Budowa modelu BiLSTM

In [ ]:
model = tf.keras.Sequential([
    tf.keras.Input(shape=(1,), dtype=tf.string),
    vectorize_layer,
    tf.keras.layers.Embedding(input_dim=MAX_TOKENS, output_dim=EMBEDDING_DIM),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(LSTM_UNITS)),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(6, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

model.summary()

## 6. Trening

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

## 7. Wykresy uczenia

In [ ]:
history_df = pd.DataFrame(history.history)
display(history_df)

plt.figure(figsize=(8, 4))
plt.plot(history.history["accuracy"], label="train accuracy")
plt.plot(history.history["val_accuracy"], label="val accuracy")
plt.xlabel("Epoka")
plt.ylabel("Accuracy")
plt.title("Accuracy w trakcie treningu")
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(history.history["loss"], label="train loss")
plt.plot(history.history["val_loss"], label="val loss")
plt.xlabel("Epoka")
plt.ylabel("Loss")
plt.title("Loss w trakcie treningu")
plt.legend()
plt.show()

## 8. Ewaluacja na zbiorze testowym

In [ ]:
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"Test accuracy: {test_acc:.4f}")

y_true = test_df["label_id"].values
y_prob = model.predict(test_ds, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

target_names = [id_to_label[i] for i in sorted(id_to_label.keys())]

print(classification_report(y_true, y_pred, target_names=target_names))

cm = confusion_matrix(y_true, y_pred)
cm_df = pd.DataFrame(cm, index=target_names, columns=target_names)
display(cm_df)

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix")
plt.ylabel("Rzeczywista klasa")
plt.xlabel("Predykcja")
plt.show()

## 9. Predykcja na własnych zdaniach

In [ ]:
samples = tf.constant([
    "Cieszę się po zdanym egzaminie i mam świetny nastrój.",
    "Jest mi smutno po tej wiadomości i nie mam na nic siły.",
    "Jestem wściekły przez tę niesprawiedliwą ocenę.",
    "Boję się jutrzejszego egzaminu i serce bije mi szybciej.",
    "To mnie obrzydza, aż odechciewa mi się jeść.",
    "Ale niespodzianka, tego się nie spodziewałem."
])

probs = model.predict(samples, verbose=0)
pred_ids = np.argmax(probs, axis=1)

results = []
for text, pred_id, prob in zip(samples.numpy(), pred_ids, probs):
    results.append({
        "text": text.decode("utf-8"),
        "predicted_label": id_to_label[int(pred_id)],
        "confidence": float(np.max(prob))
    })

display(pd.DataFrame(results))

## 10. Zapis modelu

In [ ]:
model.save("model_emocje_tf.keras")
print("Zapisano model do pliku: model_emocje_tf.keras")